# Backtest AI direction_5d

Notebook danh gia threshold, equity curve, phan tich theo nganh/thang va ket luan su dung score.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')

if 'test' not in globals():
    test = pd.read_csv('test_predictions.csv', parse_dates=['trading_date'])

test = test.copy()
test['trading_date'] = pd.to_datetime(test['trading_date'])
test['month'] = test['trading_date'].dt.to_period('M').astype(str)
if 'industry' not in test.columns:
    test['industry'] = 'Unknown'

In [ ]:
def simple_sharpe(series):
    series = pd.Series(series).dropna()
    if len(series) < 2 or series.std() == 0:
        return np.nan
    return (series.mean() / series.std()) * np.sqrt(252 / 5)

threshold_rows = []
for threshold in np.arange(0.50, 0.81, 0.05):
    trades = test[test['ai_score'] >= round(threshold, 2)].copy()
    threshold_rows.append({
        'threshold': round(threshold, 2),
        'buy_signals': len(trades),
        'win_rate': trades['direction_5d'].mean() * 100 if len(trades) else np.nan,
        'avg_return': trades['return_5d'].mean() * 100 if len(trades) else np.nan,
        'sharpe': simple_sharpe(trades['return_5d']),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.lineplot(data=threshold_df, x='threshold', y='win_rate', marker='o', ax=axes[0])
sns.lineplot(data=threshold_df, x='threshold', y='avg_return', marker='o', ax=axes[1])
sns.lineplot(data=threshold_df, x='threshold', y='sharpe', marker='o', ax=axes[2])
axes[0].set_title('Win rate')
axes[1].set_title('Average return (%)')
axes[2].set_title('Simple Sharpe')
plt.tight_layout()
plt.show()

In [ ]:
strategy = test[test['ai_score'] > 0.65].copy()
daily_strategy = strategy.groupby('trading_date')['return_5d'].mean().fillna(0)
equity = (1 + daily_strategy).cumprod()

date_index = pd.Index(sorted(test['trading_date'].unique()))
vn_index_curve = pd.Series((1 + 0.12 / 252), index=date_index).cumprod()
random_curve = pd.Series((1 + np.random.default_rng(42).normal(loc=0.0, scale=0.01, size=len(date_index))), index=date_index).cumprod()

plt.figure(figsize=(14, 6))
plt.plot(equity.index, equity.values, label='AI score > 0.65')
plt.plot(vn_index_curve.index, vn_index_curve.values, label='Buy & hold VN-Index 12%/nam')
plt.plot(random_curve.index, random_curve.values, label='Random 50%')
plt.title('Equity curve')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
industry_perf = (
    strategy.groupby('industry')
    .agg(win_rate=('direction_5d', 'mean'), avg_return=('return_5d', 'mean'), trades=('symbol', 'size'))
    .query('trades >= 10')
    .sort_values('win_rate', ascending=False)
)
display(industry_perf.head(20))

monthly_perf = strategy.groupby('month').agg(win_rate=('direction_5d', 'mean'), trades=('symbol', 'size'))
display(monthly_perf)

wrong_symbols = (
    strategy.groupby('symbol')['direction_5d']
    .mean()
    .sort_values()
    .head(10)
)
right_symbols = (
    strategy.groupby('symbol')['direction_5d']
    .mean()
    .sort_values(ascending=False)
    .head(10)
)
display(wrong_symbols.to_frame('win_rate_sai_nhieu'))
display(right_symbols.to_frame('win_rate_dung_nhieu'))

In [ ]:
market_regime = test.groupby('trading_date')['return_5d'].mean().rename('market_return_5d').reset_index()
strategy_regime = strategy.merge(market_regime, on='trading_date', how='left')
strategy_regime['market_state'] = np.where(strategy_regime['market_return_5d'] >= 0, 'Thi truong tang', 'Thi truong giam')
regime_stats = strategy_regime.groupby('market_state').agg(
    win_rate=('direction_5d', 'mean'),
    avg_return=('return_5d', 'mean'),
    trades=('symbol', 'size')
)
display(regime_stats)

In [ ]:
best_row = threshold_df.sort_values(['sharpe', 'win_rate', 'avg_return'], ascending=False).iloc[0]
worst_industries = industry_perf.sort_values('win_rate').head(5).index.tolist() if not industry_perf.empty else []

print('Ket luan:')
print(f"- Nguong score nen uu tien: {best_row['threshold']:.2f}")
print(f"- Nen tranh cac nganh co win rate thap: {', '.join(worst_industries) if worst_industries else 'Chua du du lieu'}")
print('- Gioi han cua model: chi danh gia huong 5 ngay, de bi anh huong boi thanh khoan, regime thi truong va label noise.')
print('- Can backtest them voi phi giao dich, slippage va gioi han so ma/lenh moi ngay truoc khi dung thuc te.')